# Evaluation and statistical comparison

In [ ]:
import math, re, tempfile, zipfile
from pathlib import Path
import matplotlib.pyplot as plt, numpy as np, pandas as pd, seaborn as sns
import scikit_posthocs as sp
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scipy import stats

PAPER_STYLE = {
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "lines.linewidth": 1.6,
    "patch.linewidth": 0.45,
}

DATASET_LABELS = {
    "doris": "DORIS",
    "itm": "ITM-Rec",
    "itm-rec": "ITM-Rec",
    "mars": "MARS",
}
METRIC_ORDER = ["hit", "recall", "ndcg", "map", "mrr", "precision"]
MODEL_ORDER = [
    "EDuRec",
    "SASRec",
    "BERT4Rec",
    "SGL",
    "MultiVAE",
    "LightGCN",
    "NeuMF",
    "ItemKNN",
    "UPGPR",
]


def discover_csv_files(input_path):
    p = input_path.expanduser().resolve()
    if p.is_file() and p.suffix.lower() == ".csv":
        return [p]
    if p.is_dir():
        return sorted(p.glob("*.csv"))
    if p.is_file() and p.suffix.lower() == ".zip":
        tmp = Path(tempfile.mkdtemp(prefix="eval_csvs_"))
        with zipfile.ZipFile(p) as zf:
            zf.extractall(tmp)
        return sorted(tmp.rglob("*.csv"))
    raise FileNotFoundError(f"No encuentro CSVs en: {input_path}")


def infer_dataset_name(csv_path):
    stem = csv_path.stem.lower()
    for key, label in DATASET_LABELS.items():
        if key in stem:
            return label
    return csv_path.stem.replace("_", " ").title()


def parse_mean_std(value):
    if pd.isna(value):
        return np.nan, np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value), np.nan
    nums = re.findall(r"[-+]?\d*\.?\d+", str(value))
    return (
        (float(nums[0]), float(nums[1]))
        if len(nums) > 1
        else (float(nums[0]), np.nan)
        if nums
        else (np.nan, np.nan)
    )


def metric_sort_key(m):
    base = m.split("@", 1)[0].lower()
    return (METRIC_ORDER.index(base) if base in METRIC_ORDER else len(METRIC_ORDER), m)


def model_sort_key(m):
    return (MODEL_ORDER.index(m) if m in MODEL_ORDER else len(MODEL_ORDER), m)


def safe_name(text):
    return re.sub(r"[^a-zA-Z0-9]+", "_", text.lower()).strip("_")


def has_seed(data):
    return "seed" in data.columns and data["seed"].notna().any()


## Load and normalize

In [ ]:
def collect_results(input_path, k):
    csv_files = discover_csv_files(input_path)
    rows = []
    suffix = f"@{k}"
    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        if "model" not in df.columns:
            raise ValueError(f'{csv_path.name}: no existe la columna "model".')
        ds = infer_dataset_name(csv_path)
        metrics = [
            c
            for c in df.columns
            if c.lower().endswith(suffix) and c.split("@", 1)[0].lower() in METRIC_ORDER
        ]
        if not metrics:
            raise ValueError(f"{csv_path.name}: no encuentro metricas @{k}.")
        for _, row in df.iterrows():
            model = str(row["model"])
            seed = row["seed"] if "seed" in df.columns else np.nan
            for m in metrics:
                if "seed" in df.columns and pd.api.types.is_number(row[m]):
                    mean, std, val = float(row[m]), np.nan, float(row[m])
                else:
                    mean, std = parse_mean_std(row[m])
                    val = mean
                rows.append(
                    {
                        "dataset": ds,
                        "seed": seed,
                        "model": model,
                        "metric": m.lower(),
                        "metric_base": m.split("@", 1)[0].lower(),
                        "k": k,
                        "value": val,
                        "mean": mean,
                        "std": std,
                    }
                )
    data = pd.DataFrame(rows)
    for col, key in [("model", model_sort_key), ("metric", metric_sort_key)]:
        data[col] = pd.Categorical(
            data[col], categories=sorted(data[col].unique(), key=key), ordered=True
        )
    return data.sort_values(["dataset", "metric", "model"])


def aggregate_results(data):
    if has_seed(data):
        grp = data.groupby(
            ["dataset", "model", "metric", "metric_base", "k"], observed=False
        )
        grouped = grp.agg(
            mean=("value", "mean"),
            std=("value", lambda s: s.std(ddof=1) if len(s) > 1 else np.nan),
            n_seeds=("seed", "nunique"),
        ).reset_index()
    else:
        grouped = data.copy()
        grouped["n_seeds"] = np.nan
    return apply_categories(grouped)


def apply_categories(data):
    data = data.copy()
    for col, key in [("model", model_sort_key), ("metric", metric_sort_key)]:
        data[col] = pd.Categorical(
            data[col].astype(str),
            categories=sorted(data[col].astype(str).unique(), key=key),
            ordered=True,
        )
    return data


def exclude_models(data, models_to_exclude):
    models = [str(m) for m in models_to_exclude if str(m).strip()]
    if not models:
        return data
    missing = sorted(set(models) - set(data["model"].astype(str).unique()))
    if missing:
        print(
            f"Aviso: no se encontraron estos modelos para excluir: {', '.join(missing)}"
        )
    filtered = data[~data["model"].astype(str).isin(models)]
    if filtered.empty:
        raise ValueError("La exclusion de modelos deja el estudio sin datos.")
    return apply_categories(filtered)


## Statistical analysis

In [ ]:
def pivot_for_metric(data, metric):
    vc = "value" if "value" in data.columns else "mean"
    idx = ["dataset", "seed"] if has_seed(data) else "dataset"
    p = data[data["metric"].astype(str) == metric].pivot_table(
        index=idx, columns="model", values=vc, aggfunc="first", observed=False
    )
    return p.dropna(axis=0, how="any").dropna(axis=1, how="any")


def pivot_combined(data):
    vc = "value" if "value" in data.columns else "mean"
    idx = ["dataset", "seed", "metric"] if has_seed(data) else ["dataset", "metric"]
    return (
        data.pivot_table(
            index=idx, columns="model", values=vc, aggfunc="first", observed=False
        )
        .dropna(axis=0, how="any")
        .dropna(axis=1, how="any")
    )


def avg_ranks(values):
    return values.rank(axis=1, ascending=False, method="average").mean(axis=0)


def nemenyi_pvalues(values):
    w = values.copy()
    w.index = pd.RangeIndex(len(w), name="blocks")
    w.columns.name = None
    return sp.posthoc_nemenyi_friedman(w)


def critical_distance(n_models, n_blocks, alpha):
    q = stats.studentized_range.ppf(1 - alpha, n_models, np.inf) / math.sqrt(2)
    return float(q * math.sqrt(n_models * (n_models + 1) / (6 * n_blocks)))


def friedman_for_values(values, label):
    arrays = [values[c].to_numpy(dtype=float) for c in values.columns]
    stat, pv = stats.friedmanchisquare(*arrays)
    ranks = avg_ranks(values)
    return {
        "comparison": label,
        "n_blocks": int(values.shape[0]),
        "n_models": int(values.shape[1]),
        "friedman_statistic": float(stat),
        "friedman_pvalue": float(pv),
        "best_average_rank": str(ranks.sort_values().index[0]),
        "best_rank_value": float(ranks.min()),
    }


def run_friedman_tests(data):
    rows = []
    for m in data["metric"].cat.categories:
        v = pivot_for_metric(data, str(m))
        if v.shape[0] >= 2 and v.shape[1] >= 3:
            r = friedman_for_values(v, str(m))
            r["block_type"] = "dataset_seed" if has_seed(data) else "dataset"
            rows.append(r)
    comb = pivot_combined(data)
    if comb.shape[0] >= 2 and comb.shape[1] >= 3:
        r = friedman_for_values(comb, f"all_metrics@{data['k'].iloc[0]}")
        r["block_type"] = "dataset_seed_metric" if has_seed(data) else "dataset_metric"
        rows.append(r)
    return pd.DataFrame(rows)


def run_pairwise_tests(data, proposed_model):
    rows = []
    for m in data["metric"].cat.categories:
        values = pivot_for_metric(data, str(m))
        if proposed_model not in values.columns:
            continue
        prop = values[proposed_model].astype(float)
        for model in values.columns:
            model = str(model)
            if model == proposed_model:
                continue
            base = values[model].astype(float)
            diff = prop - base
            nz = diff[np.abs(diff) > 1e-15]
            if len(nz) == 0:
                sg, pg, s2, p2 = 0.0, 1.0, 0.0, 1.0
            else:
                sg, pg = stats.wilcoxon(
                    prop,
                    base,
                    alternative="greater",
                    zero_method="wilcox",
                    method="auto",
                )
                s2, p2 = stats.wilcoxon(
                    prop,
                    base,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto",
                )
            sd = float(diff.std(ddof=1)) if len(diff) > 1 else np.nan
            cz = float(diff.mean() / sd) if sd and np.isfinite(sd) else np.nan
            rows.append(
                {
                    "metric": str(m),
                    "proposed_model": proposed_model,
                    "baseline": model,
                    "n_blocks": len(diff),
                    "block_type": "dataset_seed" if has_seed(data) else "dataset",
                    "mean_diff": float(diff.mean()),
                    "median_diff": float(diff.median()),
                    "wins": int((diff > 0).sum()),
                    "ties": int((np.abs(diff) <= 1e-15).sum()),
                    "losses": int((diff < 0).sum()),
                    "wilcoxon_statistic_greater": float(sg),
                    "wilcoxon_pvalue_greater": float(pg),
                    "wilcoxon_statistic_two_sided": float(s2),
                    "wilcoxon_pvalue_two_sided": float(p2),
                    "cohen_dz": cz,
                    "note": "Paired Wilcoxon over matched dataset-seed blocks."
                    if has_seed(data)
                    else "Paired Wilcoxon over dataset-level means; seed-level samples are not available.",
                }
            )
    return pd.DataFrame(rows)


def export_statistics(data, outdir, alpha, proposed_model):
    outdir.mkdir(parents=True, exist_ok=True)
    k = int(data["k"].iloc[0])
    data.to_csv(outdir / f"parsed_values_k{k}.csv", index=False)
    aggregate_results(data).to_csv(outdir / f"aggregate_values_k{k}.csv", index=False)
    run_friedman_tests(data).to_csv(outdir / f"friedman_results_k{k}.csv", index=False)
    run_pairwise_tests(data, proposed_model).to_csv(
        outdir / f"{safe_name(proposed_model)}_pairwise_wilcoxon_k{k}.csv", index=False
    )
    for m in data["metric"].cat.categories:
        v = pivot_for_metric(data, str(m))
        if v.shape[0] >= 2 and v.shape[1] >= 3:
            nemenyi_pvalues(v).to_csv(
                outdir / f"nemenyi_pvalues_{safe_name(str(m))}.csv"
            )
    comb = pivot_combined(data)
    if comb.shape[0] >= 2 and comb.shape[1] >= 3:
        nemenyi_pvalues(comb).to_csv(outdir / f"nemenyi_pvalues_all_metrics_k{k}.csv")
    pd.DataFrame(
        [
            {
                "alpha": alpha,
                "proposed_model": proposed_model,
                "k": k,
                "statistical_scope": "Friedman/Nemenyi use dataset-seed blocks"
                if has_seed(data)
                else "Friedman/Nemenyi use datasets as blocks",
                "limitation": "Seeds available and used as paired blocks."
                if has_seed(data)
                else "Input files contain aggregated values.",
            }
        ]
    ).to_csv(outdir / f"statistical_notes_k{k}.csv", index=False)


## Visualization

In [ ]:
def savefig(outdir, name, formats, dpi=300):
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        plt.savefig(outdir / f"{name}.{fmt}", bbox_inches="tight", dpi=dpi)


def apply_plot_style():
    sns.set_theme(
        context="paper",
        style="whitegrid",
        palette="colorblind",
        font="DejaVu Sans",
        rc=PAPER_STYLE,
    )


def style_axis(ax, grid_axis="y"):
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color="0.88", linewidth=0.7)
    ax.grid(axis="x" if grid_axis == "y" else "y", visible=False)
    sns.despine(ax=ax, trim=True)
    ax.tick_params(length=3, width=0.7, pad=5)


PROPOSED_C = "#D55E00"


def model_palette(models, proposed_model, bar=False):
    if bar:
        colors = [
            "#E4572E",
            "#1F77B4",
            "#2CA02C",
            "#9467BD",
            "#FF7F0E",
            "#17BECF",
            "#D62728",
            "#8C564B",
            "#BCBD22",
            "#E377C2",
            "#4C78A8",
            "#72B7B2",
        ]
        proposed_color = colors[0]
        base = [c for c in colors[1:] if c != proposed_color]
    else:
        proposed_color = PROPOSED_C
        base = [
            to_hex(c).upper()
            for c in sns.color_palette("colorblind", max(len(models) + 2, 10))
            if to_hex(c).upper() != proposed_color
        ]
    pal, idx = {}, 0
    for m in models:
        pal[m] = proposed_color if m == proposed_model else base[idx % len(base)]
        idx += 0 if m == proposed_model else 1
    return pal


def display_label(model, proposed_model):
    return "Propuesta" if str(model) == proposed_model else str(model)


def plot_grouped_bars(data, outdir, formats, proposed_model, show_error_bars=False):
    for metric in data["metric"].cat.categories:
        subset = data[data["metric"].astype(str) == metric]
        if subset.empty:
            continue
        apply_plot_style()
        fig, ax = plt.subplots(figsize=(7.4, 4.8), layout="constrained")
        models = sorted(subset["model"].astype(str).unique(), key=model_sort_key)
        pal = model_palette(models, proposed_model, bar=True)
        sns.barplot(
            data=subset,
            x="dataset",
            y="mean",
            hue="model",
            hue_order=models,
            errorbar=None,
            palette=pal,
            edgecolor="0.25",
            linewidth=0.45,
            ax=ax,
        )
        if show_error_bars:
            ordered = []
            for ds in [t.get_text() for t in ax.get_xticklabels()]:
                ordered.extend(
                    subset[subset["dataset"] == ds]
                    .set_index("model")
                    .reindex(models)
                    .reset_index()
                    .to_dict("records")
                )
            for patch, row in zip(
                [p for p in ax.patches if p.get_width() > 0], ordered
            ):
                if pd.notna(row.get("std")) and pd.notna(row.get("mean")):
                    ax.errorbar(
                        patch.get_x() + patch.get_width() / 2,
                        row["mean"],
                        yerr=row["std"],
                        color="0.18",
                        capsize=1.8,
                        elinewidth=0.65,
                        capthick=0.65,
                        fmt="none",
                        zorder=4,
                    )
        ax.set_title(f"Comparacion por conjunto de datos ({metric.upper()})", pad=10)
        ax.set_ylabel(metric.upper())
        style_axis(ax, "y")
        h, l = ax.get_legend_handles_labels()
        ax.legend(
            h,
            [display_label(lb, proposed_model) for lb in l],
            title=None,
            frameon=False,
            ncol=3,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.16),
            borderaxespad=0.0,
        )
        savefig(outdir, f"bar_comparison_{safe_name(str(metric))}", formats)
        plt.close(fig)


def plot_metric_grid(data, outdir, formats, proposed_model):
    apply_plot_style()
    metrics = list(data["metric"].cat.categories)
    ncols, nrows = 3, math.ceil(len(metrics) / 3)
    fig, axes = plt.subplots(nrows, ncols, figsize=(10.2, 3.5 * nrows))
    axes = np.asarray(axes).reshape(-1)
    models = sorted(data["model"].astype(str).unique(), key=model_sort_key)
    pal = model_palette(models, proposed_model)
    for ax, m in zip(axes, metrics):
        subset = data[data["metric"].astype(str) == str(m)]
        sns.lineplot(
            data=subset,
            x="dataset",
            y="mean",
            hue="model",
            style="model",
            hue_order=models,
            markers=True,
            dashes=False,
            errorbar=None,
            palette=pal,
            ax=ax,
            legend=False,
        )
        ax.set_title(m.upper())
        style_axis(ax, "y")
    for ax in axes[len(metrics) :]:
        ax.axis("off")
    fig.subplots_adjust(
        left=0.06, right=0.99, top=0.94, bottom=0.20, hspace=0.50, wspace=0.18
    )
    handles = [
        Line2D(
            [0],
            [0],
            color=pal[m],
            marker="o",
            linewidth=1.8,
            label=display_label(m, proposed_model),
        )
        for m in models
    ]
    if handles:
        fig.legend(
            handles,
            [h.get_label() for h in handles],
            loc="lower center",
            ncol=min(5, len(models)),
            frameon=False,
            bbox_to_anchor=(0.5, 0.03),
        )
    savefig(outdir, f"metric_profile_k{int(data['k'].iloc[0])}", formats)
    plt.close(fig)


def relative_improvement_table(data, proposed_model, baseline):
    rows = []
    for m in data["metric"].cat.categories:
        values = pivot_for_metric(data, str(m))
        if proposed_model not in values.columns:
            continue
        for ds, row in values.iterrows():
            if baseline == "best-baseline":
                cand = row.drop(labels=[proposed_model], errors="ignore")
                bl_model, bl_val = str(cand.idxmax()), float(cand.max())
            else:
                if baseline not in row.index:
                    continue
                bl_model, bl_val = baseline, float(row[baseline])
            pv = float(row[proposed_model])
            impr = 100.0 * (pv - bl_val) / abs(bl_val) if bl_val != 0 else np.nan
            rows.append(
                {
                    "dataset": ds,
                    "metric": str(m),
                    "proposed_model": proposed_model,
                    "baseline": bl_model,
                    "proposed_value": pv,
                    "baseline_value": bl_val,
                    "relative_improvement_percent": impr,
                }
            )
    return pd.DataFrame(rows)


def plot_relative_improvement(data, outdir, formats, proposed_model, baseline):
    impr = relative_improvement_table(data, proposed_model, baseline)
    k = int(data["k"].iloc[0])
    tdir = Path(globals().get("TABLES_DIR", outdir))
    tdir.mkdir(parents=True, exist_ok=True)
    impr.to_csv(
        tdir / f"relative_improvement_{safe_name(proposed_model)}_k{k}.csv", index=False
    )
    if impr.empty:
        return
    pivot = impr.pivot_table(
        index="dataset",
        columns="metric",
        values="relative_improvement_percent",
        aggfunc="first",
        observed=False,
    ).reindex(columns=list(data["metric"].cat.categories))
    apply_plot_style()
    fig, ax = plt.subplots(figsize=(7.2, 3.0), layout="constrained")
    vmax = max(float(np.nanmax(np.abs(pivot.to_numpy(dtype=float)))), 1.0)
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1f",
        cmap="vlag",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Mejora relativa (%)"},
        ax=ax,
    )
    ax.set_title(
        f"Mejora relativa de {display_label(proposed_model, proposed_model)} frente a {baseline}"
    )
    savefig(
        outdir,
        f"relative_improvement_heatmap_{safe_name(proposed_model)}_k{k}",
        formats,
    )
    plt.close(fig)


def plot_rank_heatmap(data, outdir, formats, proposed_model):
    ranks = []
    for m in data["metric"].cat.categories:
        values = pivot_for_metric(data, str(m))
        ranked = values.rank(axis=1, ascending=False, method="average")
        for ds, row in ranked.iterrows():
            for model, rank in row.items():
                ranks.append(
                    {"dataset": ds, "metric": str(m), "model": str(model), "rank": rank}
                )
    rank_df = pd.DataFrame(ranks)
    k = int(data["k"].iloc[0])
    tdir = Path(globals().get("TABLES_DIR", outdir))
    tdir.mkdir(parents=True, exist_ok=True)
    rank_df.to_csv(tdir / f"ranks_k{k}.csv", index=False)
    pivot = rank_df.pivot_table(
        index="model",
        columns=["dataset", "metric"],
        values="rank",
        aggfunc="first",
        observed=False,
    )
    pivot = pivot.reindex(index=sorted(pivot.index, key=model_sort_key))
    pivot.index = [display_label(m, proposed_model) for m in pivot.index]
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(9.0, 0.42 * pivot.shape[1] + 3.0), 4.8), layout="constrained"
    )
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1f",
        cmap="viridis_r",
        linewidths=0.35,
        linecolor="white",
        cbar_kws={"label": "Ranking medio (1 = mejor)"},
        ax=ax,
    )
    ax.set_title(f"Rankings por conjunto de datos y metrica (@{k})")
    ax.tick_params(axis="x", labelrotation=45)
    savefig(outdir, f"rank_heatmap_k{k}", formats)
    plt.close(fig)


def plot_cd_diagram_for_values(
    values, title, outdir, name, formats, alpha, proposed_model
):
    ranks = avg_ranks(values).sort_values()
    pv = nemenyi_pvalues(values).reindex(index=ranks.index, columns=ranks.index)
    cd = critical_distance(values.shape[1], values.shape[0], alpha)
    labels = [display_label(m, proposed_model) for m in ranks.index]
    ranks.index = labels
    pv.index = labels
    pv.columns = labels
    apply_plot_style()
    fig, ax = plt.subplots(
        figsize=(max(8.8, 0.38 * len(ranks) + 7.2), max(4.0, 0.44 * len(ranks) + 1.7))
    )
    fig.suptitle(title, y=0.98, fontsize=10.5, fontweight="normal")
    fig.subplots_adjust(left=0.18, right=0.82, top=0.88, bottom=0.10)
    hl = display_label(proposed_model, proposed_model)
    colors = {str(m): "#D55E00" if str(m) == hl else "#128CBD" for m in ranks.index}
    sp.critical_difference_diagram(
        ranks,
        pv,
        cd=cd,
        alpha=alpha,
        ax=ax,
        color_palette=colors,
        label_fmt_left="{label}\n({rank:.2f})",
        label_fmt_right="{label}\n({rank:.2f})",
        label_props={
            "fontsize": 9.0,
            "linespacing": 1.25,
            "bbox": {"facecolor": "white", "edgecolor": "none", "pad": 1.5, "alpha": 0.96},
        },
        marker_props={"s": 26, "zorder": 3},
        elbow_props={"linewidth": 1.15},
        crossbar_props={"linewidth": 1.6, "color": "0.12"},
        text_h_margin=0.06,
    )
    ax.tick_params(axis="x", which="major", pad=12, labelsize=9)
    for t in ax.get_xticklabels():
        t.set_bbox({"facecolor": "white", "edgecolor": "none", "pad": 0.35, "alpha": 0.96})
    for t in ax.texts:
        if t.get_text().startswith("CD = "):
            t.set(
                fontsize=9,
                y=1.12,
                bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.4, "alpha": 0.98},
            )
    if ax.get_ylim()[1] < 1.28:
        ax.set_ylim(ax.get_ylim()[0], 1.28)
    savefig(outdir, name, formats)
    plt.close(fig)


def plot_cd_diagrams(data, outdir, formats, alpha, proposed_model):
    for m in data["metric"].cat.categories:
        v = pivot_for_metric(data, str(m))
        if v.shape[0] >= 2 and v.shape[1] >= 3:
            plot_cd_diagram_for_values(
                v,
                f"Diagrama de diferencia critica ({m.upper()}, alfa={alpha})",
                outdir,
                f"cd_diagram_{safe_name(str(m))}",
                formats,
                alpha,
                proposed_model,
            )
    k = int(data["k"].iloc[0])
    comb = pivot_combined(data)
    if comb.shape[0] >= 2 and comb.shape[1] >= 3:
        plot_cd_diagram_for_values(
            comb,
            f"Diagrama de diferencia critica (todas las metricas @{k}, alfa={alpha})",
            outdir,
            f"cd_diagram_all_metrics_k{k}",
            formats,
            alpha,
            proposed_model,
        )


## Configuration

In [ ]:
def find_project_root(start=Path.cwd()):
    for c in (start.resolve(), *start.resolve().parents):
        if (c / "plots").is_dir() and (c / "notebooks").is_dir():
            return c
    raise FileNotFoundError("Could not locate the EDuRec repository root.")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "plots" / "evaluations"
PLOTS_DIR = PROJECT_ROOT / "results" / "plots" / "all_datasets"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "evaluation"
K = 20
PROPOSED_MODEL = "EDuRec"
BASELINE_FOR_IMPROVEMENT = "best-baseline"
EXCLUDE_MODELS = []
ALPHA = 0.05
FORMATS = ["png"]
SHOW_ERROR_BARS = False

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_PATH}")
print(f"Plots:  {PLOTS_DIR}")
print(f"Tables: {TABLES_DIR}")


## Load, validate, aggregate

In [ ]:
results = collect_results(INPUT_PATH, K)
if PROPOSED_MODEL in EXCLUDE_MODELS:
    raise ValueError("The proposed model cannot also be excluded.")
results = exclude_models(results, EXCLUDE_MODELS)
if PROPOSED_MODEL not in results["model"].astype(str).unique():
    raise ValueError(f"Proposed model {PROPOSED_MODEL!r} is absent from the input.")
plot_data = aggregate_results(results)
print(
    f"Loaded {len(results)} normalized observations across "
    f"{results['dataset'].nunique()} datasets, "
    f"{results['model'].nunique()} models, and "
    f"{results['metric'].nunique()} metrics."
)
plot_data.head()


## Export statistics

In [ ]:
export_statistics(results, TABLES_DIR, ALPHA, PROPOSED_MODEL)
run_friedman_tests(results)


## Generate figures

In [ ]:
plot_grouped_bars(
    plot_data, PLOTS_DIR, FORMATS, PROPOSED_MODEL, show_error_bars=SHOW_ERROR_BARS
)
plot_metric_grid(plot_data, PLOTS_DIR, FORMATS, PROPOSED_MODEL)
plot_relative_improvement(
    plot_data,
    PLOTS_DIR,
    FORMATS,
    proposed_model=PROPOSED_MODEL,
    baseline=BASELINE_FOR_IMPROVEMENT,
)
plot_rank_heatmap(plot_data, PLOTS_DIR, FORMATS, PROPOSED_MODEL)
plot_cd_diagrams(
    results, PLOTS_DIR, FORMATS, alpha=ALPHA, proposed_model=PROPOSED_MODEL
)

files = sorted(p.name for p in PLOTS_DIR.iterdir() if p.is_file())
print(f"Generated {len(files)} files in {PLOTS_DIR}")
files
